# Week 8-B: Generator Draft State and Uncertainty Diagnostics
## Diagnosing Generator Information Needs from Base Draft $y_0$

In DAFiR, routing is **generator-aware**. Before promoting any evidence, we inspect the base draft $y_0$ generated under semantic $k=4$ ($4F + 6H$):
1. **Token Uncertainty Features ($U_1 \dots U_{12}$)**: Shannon entropy, top1-top2 margin, sequence logprob, EOS confidence.
2. **Sufficiency Classifiers ($B_0, B_1, B_2$)**: Logistic, GBDT, and MLP models predicting whether base draft is sufficient ($F1_0 \ge 0.8$).
3. **Calibration & Expected Calibration Error (ECE)**.
4. **Stochastic Self-Consistency ($N=3$)**: Token Jaccard disagreement and majority voting.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
SARA_ROOT = REPO_ROOT / "Baselines" / "SARA-main"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src
if hasattr(src, "__path__") and str(SARA_ROOT / "src") not in src.__path__:
    src.__path__.append(str(SARA_ROOT / "src"))

from src.model.loader import load_sara_for_eval
from src.week8.semantic import SemanticScorer
from src.week8.data import load_config, load_week8_split
from src.week8.draft_generation import execute_resumable_draft_generation
from src.week8.self_consistency import run_self_consistency_sampling
from src.week8.uncertainty import (
    compute_uncertainty_features_for_row,
    evaluate_sufficiency_models,
    compute_expected_calibration_error,
)

cfg = load_config()
split = "internal_val"
records, sfr_dict, _ = load_week8_split(split, cfg)
scorer = SemanticScorer()

device = "cuda" if torch.cuda.is_available() else "cpu"
sara_ckpt_path = REPO_ROOT / cfg["sara"]["checkpoint_path"]
sara_model, sara_tokenizer = load_sara_for_eval(str(sara_ckpt_path), device=device)
print(f"Loaded SARA generator on {device} for split '{split}' ({len(records)} queries).")


In [ ]:
# Compute semantic similarities
semantic_scores_by_qid = {}
for r in records:
    qid = str(r["question_id"])
    sims = scorer.score_query_passages(r["question"], [p["text"] for p in r.get("passages", [])[:10]]).cpu().numpy()
    semantic_scores_by_qid[qid] = sims

# Execute Draft Generation with Diagnostics (y0)
draft_cache_path = REPO_ROOT / "results/week8/drafts" / f"draft_diagnostics_{split}.jsonl"
df_drafts = execute_resumable_draft_generation(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=draft_cache_path,
    device=device,
)
print(f"Draft generation complete: {len(df_drafts)} drafts with per-step token diagnostics.")


In [ ]:
# Extract 12 Draft Uncertainty Features (U1..U12)
u_rows = []
for _, row in df_drafts.iterrows():
    u_dict = compute_uncertainty_features_for_row(row.to_dict(), semantic_scorer=scorer)
    u_dict["question_id"] = str(row["question_id"])
    u_dict["base_f1"] = float(row.get("base_f1", 0.0))
    u_rows.append(u_dict)

df_u = pd.DataFrame(u_rows)
u_feature_cols = [c for c in df_u.columns if c.startswith("u_")]
print(f"Extracted {len(u_feature_cols)} uncertainty features:")
print(df_u[u_feature_cols].describe().T[["mean", "std", "min", "max"]])


In [ ]:
# Evaluate Sufficiency Classifiers (B0, B1, B2)
# Target: Is draft sufficient (base_f1 >= 0.8 or safe-stop F1=1.0)?
X = df_u[u_feature_cols].values
y = (df_u["base_f1"] >= 0.8).astype(int).values

results_suff = evaluate_sufficiency_models(X, y)
print("=== Sufficiency Classifier Evaluation (Predicting Draft Sufficiency) ===")
for model_name, metrics in results_suff.items():
    print(f"  {model_name}: Accuracy = {metrics['accuracy']:.4f}, ROC-AUC = {metrics['roc_auc']:.4f}, ECE = {metrics['ece']:.4f}")


In [ ]:
# Stochastic Self-Consistency Sampling (N=3, T=0.7, top_p=0.9)
sc_cache_path = REPO_ROOT / "results/week8/drafts" / f"self_consistency_{split}.jsonl"
df_sc = run_self_consistency_sampling(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=sc_cache_path,
    n_samples=3,
    temperature=0.7,
    top_p=0.9,
    device=device,
)
print(f"Self-consistency sampling complete for {len(df_sc)} queries.")
print(f"Mean Pairwise Jaccard Agreement: {df_sc['sc_mean_pairwise_jaccard'].mean():.4f}")
print(f"Sample Disagreement Rate:        {df_sc['sc_disagreement_indicator'].mean():.4f}")
